In [21]:
import os
from pathlib import Path
from openai import OpenAI
import sys
import base64
from dotenv import load_dotenv
import json
import mimetypes
import subprocess

In [22]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    from google.colab import userdata
    api_key = userdata.get("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY is missing.")

print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [23]:
repo_url = "https://github.com/ShRuwali/Physics_LLM_Test.git"
repo_dir = Path("/content/Physics_LLM_Test")

if repo_dir.exists():
    subprocess.run(
        ["git", "-C", str(repo_dir), "pull", "origin", "main"],
        check=True
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", "main", repo_url, str(repo_dir)],
        check=True
    )

In [24]:

import sys
from pathlib import Path

repo_dir = Path("/content/Physics_LLM_Test")
video_dir = repo_dir / "Problems" / "Video"

# Check that GitHub files were downloaded
print("Repository exists:", repo_dir.exists())
print("Video folder exists:", video_dir.exists())
print("Prompt file exists:", (video_dir / "video_prompts.py").exists())

# Import video prompts directly from folder
sys.path.insert(0, str(video_dir))

from video_prompts import regular_system_prompt
from video_prompts import detail_system_prompt

print("Regular prompts:", len(regular_system_prompt))
print("Detailed prompts:", len(detail_system_prompt))
print("Total prompts:", len(regular_system_prompt) + len(detail_system_prompt))

# Print all regular system prompts
print("\nREGULAR SYSTEM PROMPTS:")
print(regular_system_prompt)

# Print all detailed system prompts
print("\nDETAILED SYSTEM PROMPTS:")
print(detail_system_prompt)


Repository exists: True
Video folder exists: True
Prompt file exists: True
Regular prompts: 24
Detailed prompts: 16
Total prompts: 40

REGULAR SYSTEM PROMPTS:
{'Problem_1_Regular_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video. Format your answer as Markdown. Use LaTeX for all mathematical notation.Use $...$ for inline equations. For display equations, put $$ on its own line before and after the equation. Do not wrap the response or equations in backticks or code fences.Use one backslash per LaTeX command, such as \\frac or \\sqrt.', 'Problem_1_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video. Format your answer as Markdown. Use LaTeX for all mathematical notation.Use $...$ for inli

In [25]:
print(regular_system_prompt.keys())
print(detail_system_prompt.keys())

# Path to video files
video_folder = repo_dir / "Problems" / "Video"

# List the video filenames
file_names = sorted(file.name for file in video_folder.glob("*.mp4"))

print("Total number of files:::", len(file_names))
print(file_names)

## collect the dictionary to loop through
prompt_dict = [regular_system_prompt, detail_system_prompt]

## connect to OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

# Match video filenames with prompt keys regardless of capitalization
video_lookup = {
    Path(file).stem.lower(): file
    for file in file_names
}

# Check that every prompt has a corresponding video
for prompts in prompt_dict:
    for problem_name in prompts:
        if problem_name.lower() not in video_lookup:
            print("Missing video:", problem_name)

print("Video matching check completed.")


dict_keys(['Problem_1_Regular_Q', 'Problem_1_Obvious_Q', 'Problem_1_non_Obvious_Q', 'Problem_2_Regular_Q', 'Problem_2_Obvious_Q', 'Problem_2_non_obvious_Q', 'Problem_3_Regular_Q', 'Problem_3_Obvious_Q', 'Problem_3_non_Obvious_Q', 'Problem_4_Regular_Q', 'Problem_4_Obvious_Q', 'Problem_4_non_obvious_Q', 'Problem_5_Regular_Q', 'Problem_5_Obvious_Q', 'Problem_5_non_obvious_Q', 'Problem_6_Regular_Q', 'Problem_6_Obvious_Q', 'Problem_6_non_obvious_Q', 'Problem_7_Regular_Q', 'Problem_7_Obvious_Q', 'Problem_7_non_obvious_Q', 'Problem_8_Regular_Q', 'Problem_8_Obvious_Q', 'Problem_8_non_obvious_Q'])
dict_keys(['Problem_1_Obvious_Q', 'Problem_1_non_Obvious_Q', 'Problem_2_Obvious_Q', 'Problem_2_non_obvious_Q', 'Problem_3_Obvious_Q', 'Problem_3_non_Obvious_Q', 'Problem_4_Obvious_Q', 'Problem_4_non_obvious_Q', 'Problem_5_Obvious_Q', 'Problem_5_non_obvious_Q', 'Problem_6_Obvious_Q', 'Problem_6_non_obvious_Q', 'Problem_7_Obvious_Q', 'Problem_7_non_obvious_Q', 'Problem_8_Obvious_Q', 'Problem_8_non_obvio

In [26]:
# Match video filenames with prompt keys regardless of capitalization
video_lookup = {
    Path(file).stem.lower(): file
    for file in file_names
}

# Check that every prompt has a corresponding video
for prompts in prompt_dict:
    for problem_name in prompts:
        if problem_name.lower() not in video_lookup:
            print("Missing video:", problem_name)

print("Video matching check completed.")


Video matching check completed.


In [29]:
%%time

## Here the filename and dictionary key must match.
## Capitalization differences are handled.

meta_data = {}  ## useful for creating metadata stored as a json file

# Gemini 3.6 Flash
Models = ["google/gemini-3.6-flash"]

# Path to video files
video_folder = repo_dir / "Problems" / "Video"

# Path to results
folder = repo_dir / "Result" / "frontier_models" / "Video" / "gemini_3.6"
output_folder = folder / "metadata"

folder.mkdir(parents=True, exist_ok=True)
output_folder.mkdir(parents=True, exist_ok=True)

completed = 0
skipped = 0
failed = []

for model in Models:
    for dicts in prompt_dict:
        for file in file_names:

            ## First read the video files
            file_path = video_folder / file

            ## Skip detailed system prompt for regular questions
            if dicts is detail_system_prompt and "regular" in file_path.stem.lower():
                continue

            file_key = file_path.stem

            ## Match filename with dictionary key (case-insensitive)
            matched_key = next(
                (key for key in dicts if key.lower() == file_key.lower()),
                None
            )

            if matched_key is None:
                print(f"For {file_key} no dictionary key was found. Some error in file naming.")
                continue

            # Use exact dictionary key for consistent output naming
            file_key = matched_key
            system_prompt = dicts[file_key]

            model_name = model.split("/")[-1]

            if dicts is detail_system_prompt:
                end_name = "detail_system_prompt"
            else:
                end_name = "regular_system_prompt"

            output_filename = f"{model_name}_{file_key}_{end_name}.md"
            output_file_path = folder / output_filename

            output_name = f"{model_name}_{file_key}_{end_name}.json"
            meta_data_file_path = output_folder / output_name

            ## Skip evaluations that already have both files
            if output_file_path.exists() and meta_data_file_path.exists():
                print(f"SKIPPED:::{model_name}_{file_key}_{end_name}")
                skipped += 1
                continue

            try:
                ## Encode MP4 video as base64
                base64_video = base64.b64encode(
                    file_path.read_bytes()
                ).decode("utf-8")

                mime_type = mimetypes.guess_type(str(file_path))[0] or "video/mp4"

                ## Send video to Gemini through OpenRouter
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text",
                                    "text": system_prompt,
                                },
                                {
                                    "type": "video_url",
                                    "video_url": {
                                        "url": f"data:{mime_type};base64,{base64_video}"
                                    },
                                },
                            ],
                        }
                    ],
                )

                output_texts = response.choices[0].message.content

                if not output_texts:
                    raise ValueError("Model returned an empty response.")

                ## Save Markdown response
                output_file_path.write_text(
                    output_texts,
                    encoding="utf-8"
                )

                ## Get token usage
                usage = getattr(response, "usage", None)

                ## Create metadata
                metadata = {
                    "requested_model": model,
                    "returned_model": getattr(response, "model", None),
                    "response_id": getattr(response, "id", None),
                    "prompt_name": file_key,
                    "system_prompt": system_prompt,
                    "generation_settings": meta_data,
                    "token_usage": {
                        "input_tokens": getattr(usage, "prompt_tokens", None),
                        "output_tokens": getattr(usage, "completion_tokens", None),
                        "total_tokens": getattr(usage, "total_tokens", None),
                    },
                }

                ## Convert dictionary to JSON
                meta_data_file_path.write_text(
                    json.dumps(metadata, indent=4, ensure_ascii=False),
                    encoding="utf-8"
                )

                completed += 1
                print(f"Completed:::{model_name}_{file_key}_{end_name}")

            except Exception as e:
                failed.append((file_key, end_name, str(e)))

                print(f"FAILED:::{model_name}_{file_key}_{end_name}")
                print("Error:", str(e))
                print("Stopping so the error can be investigated.")
                break

        if failed:
            break

    if failed:
        break

print("\n========== EVALUATION SUMMARY ==========")
print("Model:", Models[0])
print("Newly completed:", completed)
print("Previously completed:", skipped)
print("Failed:", len(failed))
print("Total Markdown files:", len(list(folder.glob("*.md"))))
print("Total JSON files:", len(list(output_folder.glob("*.json"))))


Completed:::gemini-3.6-flash_Problem_1_Obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_1_Regular_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_1_non_Obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_2_Obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_2_Regular_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_2_non_obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_3_Obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_3_Regular_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_3_non_Obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_4_Obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_4_Regular_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_4_non_obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_5_Obvious_Q_regular_system_prompt
Completed:::gemini-3.6-flash_Problem_5_Regular_Q_regular_syst